In [0]:
import pyspark.sql.functions as f

readind data into spark data frames

In [0]:
india_df = spark.read.table("azure_wak_learn.bronze.india_sales")
america_df = spark.read.table("azure_wak_learn.bronze.america_sales")
uae_df = spark.read.table("azure_wak_learn.bronze.uae_sale")

In [0]:
india_df = india_df.withColumn("Transaction_ID",
                                f.when(f.col("Transaction_ID").startswith("txn"),f.regexp_replace(f.col("Transaction_ID"), "txn","IND"))
                                 .when(f.col("Transaction_ID").startswith("1"),f.concat(f.lit("IND-"),f.col("Transaction_ID")))
                                .otherwise(f.col("Transaction_ID"))
                                )
america_df = america_df.withColumn("Transaction_ID",
                                f.when(f.col("Transaction_ID").startswith("txn"),f.regexp_replace(f.col("Transaction_ID"), "txn","AME"))
                                 .when(f.col("Transaction_ID").startswith("1"),f.concat(f.lit("AME-"),f.col("Transaction_ID")))
                                .otherwise(f.col("Transaction_ID"))
                                )
e_df = uae_df.withColumn("Transaction_ID",
                                f.when(f.col("Transaction_ID").startswith("txn"),f.regexp_replace(f.col("Transaction_ID"), "txn","UAE"))
                                 .when(f.col("Transaction_ID").startswith("DXB"),f.regexp_replace(f.col("Transaction_ID"), "DXB","UAE"))
                                .otherwise(f.col("Transaction_ID"))
                                )
uae_df = uae_df.dropDuplicates(["Transaction_ID"])

In [0]:
prefixes = "(?i)^(mr|mrs|ms|dr|miss)\\.?\\s+"
suffixes = "(?i)\\s+(phd|dds|jr|sr|md|iv|iii|ii)\\.?$"
uae_df = uae_df.withColumn("Customer_Name",
                          
                          f.trim(f.regexp_replace(f.regexp_replace(f.lower(f.col("Customer_Name")), prefixes, ""), suffixes, ""))
                ).withColumn("City",f.lower(f.trim(f.col("City")))
                ).withColumn("State",f.lower(f.trim(f.col("State")))
                ).withColumn("Country",f.lit("UAE")
                ).withColumn("Product_Name",f.lower(f.trim(f.col("Product_Name")))
                )
america_df = america_df.withColumn("Customer_Name",
                          
                          f.trim(f.regexp_replace(f.regexp_replace(f.lower(f.col("Customer_Name")), prefixes, ""), suffixes, ""))
                           ).withColumn("City",f.lower(f.trim(f.col("City")))
                ).withColumn("State",f.lower(f.trim(f.col("State")))
                ).withColumn("Country",f.lit("America")
                ).withColumn("Product_Name",f.lower(f.trim(f.col("Product_Name")))
                )
india_df = india_df.withColumn("Customer_Name",
                          
                          f.trim(f.regexp_replace(f.regexp_replace(f.lower(f.col("Customer_Name")), prefixes, ""), suffixes, ""))
                ).withColumn("City",f.lower(f.trim(f.col("City")))
                ).withColumn("State",f.lower(f.trim(f.col("State")))
                ).withColumn("Country",f.lit("India")
                ).withColumn("Product_Name",f.lower(f.trim(f.col("Product_Name")))
                )

In [0]:
uae_df = uae_df.withColumn("Quantity",
                                      f.when(f.col("Quantity").rlike("pcs"),f.regexp_replace(f.col("Quantity"), "pcs",""))
                                      .otherwise(f.col("Quantity"))
                ).withColumn("Quantity",f.trim(f.col("Quantity"))
                ).withColumn("Quantity", f.col("Quantity").cast("int"))
america_df = america_df.withColumn("Quantity",
                                      f.when(f.col("Quantity").rlike("pcs"),f.regexp_replace(f.col("Quantity"), "pcs",""))
                                      .otherwise(f.col("Quantity"))
                ).withColumn("Quantity",f.trim(f.col("Quantity"))
                ).withColumn("Quantity", f.col("Quantity").cast("int"))
india_df = india_df.withColumn("Quantity",
                                      f.when(f.col("Quantity").rlike("pcs"),f.regexp_replace(f.col("Quantity"), "pcs",""))
                                      .otherwise(f.col("Quantity"))
                ).withColumn("Quantity",f.trim(f.col("Quantity"))
                ).withColumn("Quantity", f.col("Quantity").cast("int"))

In [0]:
uae_df = uae_df.withColumn("Unit_Price",
                           f.regexp_replace(f.col("Unit_Price"), "(?i)AED", "")
               ).withColumn("Unit_Price", f.col("Unit_Price").cast("double")
               )
america_df = america_df.withColumn("Unit_Price",
                           f.regexp_replace(f.col("Unit_Price"), r"(?i)\$|USD", "")
               ).withColumn("Unit_Price", f.col("Unit_Price").cast("double")
               )

In [0]:
uae_df = uae_df.withColumn("Total_Sales",
                   f.lower(f.trim(f.regexp_replace(f.col("Total_Sales"), "(?i)AED", "")))
               ).withColumn("Total_Sales", f.col("Total_Sales").cast("double")
               )
america_df = america_df.withColumn("Total_Sales",
                   f.lower(f.trim(f.regexp_replace(f.col("Total_Sales"), r"(?i)\$|USD", "")))
               ).withColumn("Total_Sales", f.col("Total_Sales").cast("double")
               )
    

In [0]:

USD_TO_INR_RATE = 95.7
cleaned_df = india_df.withColumn("cleaned_amt",
                                 f.regexp_replace(f.col("Unit_Price"),r"[\$₹]", ""))
cleaned_df = cleaned_df.withColumn("cleaned_amt", f.col("cleaned_amt").cast("double"))
india_df = cleaned_df.withColumn(
    "Unit_Price",
    f.round(f.when(f.col("Unit_Price").contains("$"), f.col("cleaned_amt") * USD_TO_INR_RATE)
     .otherwise(f.col("cleaned_amt")),2)
)
india_df = india_df.drop(f.col("cleaned_amt"))
india_df = india_df.withColumn("Total_Sales",
                   f.lower(f.trim(f.regexp_replace(f.col("Total_Sales"), r"(?i)\₹|Rs.", "")))
               ).withColumn("Total_Sales", f.col("Total_Sales").cast("double")
               )

In [0]:
india_df = india_df.withColumn("Order_Date",f.lower(f.trim(f.col("Order_Date")))
               ).withColumn(
                    "formatted_date",
                    f.coalesce(
                    f.try_to_date("Order_Date", "MM-dd-yyyy"),
                    f.try_to_date("Order_Date", "dd/MM/yyyy"),
                    f.try_to_date("Order_Date", "yyyy-MM-dd")
            )
                ).withColumn("Order_Date",f.col("formatted_date")
                ).drop("formatted_date")
uae_df = uae_df.withColumn("Order_Date",f.lower(f.trim(f.col("Order_Date")))
               ).withColumn(
                    "formatted_date",
                    f.coalesce(
                    f.try_to_date("Order_Date", "MM-dd-yyyy"),
                    f.try_to_date("Order_Date", "dd/MM/yyyy"),
                    f.try_to_date("Order_Date", "yyyy-MM-dd")
            )
                ).withColumn("Order_Date",f.col("formatted_date")
                ).drop("formatted_date")
america_df = america_df.withColumn("Order_Date",f.lower(f.trim(f.col("Order_Date")))
               ).withColumn(
                    "formatted_date",
                    f.coalesce(
                    f.try_to_date("Order_Date", "MM-dd-yyyy"),
                    f.try_to_date("Order_Date", "dd/MM/yyyy"),
                    f.try_to_date("Order_Date", "yyyy-MM-dd")
            )
                ).withColumn("Order_Date",f.col("formatted_date")
                ).drop("formatted_date")


In [0]:
america_df = america_df.withColumn("Customer_Email",
                                    f.lower(f.regexp_replace(f.trim(f.col("Customer_Email")),r"\s+",""))
                                    )
uae_df = uae_df.withColumn("Customer_Email",
                                    f.lower(f.regexp_replace(f.trim(f.col("Customer_Email")),r"\s+",""))
                                    )
india_df = india_df.withColumn("Customer_Email",
                                    f.lower(f.regexp_replace(f.trim(f.col("Customer_Email")),r"\s+",""))
                                    )

In [0]:
america_df = america_df.withColumn("Store_ID",
                                   f.upper(f.trim(f.col("Store_ID")))
                                   )
america_df = america_df.withColumn("Store_ID",
                                   f.when(f.col("Store_ID").startswith("S"),f.col("Store_ID"))
                                   .otherwise(f.concat(f.lit("S-"),f.col("Store_ID")))
                                    )
uae_df = uae_df.withColumn("Store_ID",
                                   f.upper(f.trim(f.col("Store_ID")))
                                   )
uae_df = uae_df.withColumn("Store_ID",
                                   f.when(f.col("Store_ID").startswith("DXB"),f.col("Store_ID"))
                                   .otherwise(f.concat(f.lit("DXB-"),f.col("Store_ID")))
                                    )
india_df = india_df.withColumn("Store_ID",
                                   f.upper(f.trim(f.col("Store_ID")))
                                   )
india_df = india_df.withColumn("Store_ID",
                                   f.when(f.col("Store_ID").startswith("S"),f.col("Store_ID"))
                                   .otherwise(f.concat(f.lit("S-"),f.col("Store_ID")))
                            ).withColumn("Store_ID",
                                  f.regexp_replace("Store_ID","S","I")
                                    )
                                   

In [0]:
america_df = america_df.drop("Total_Sales")
india_df = india_df.drop("Total_Sales")
uae_df = uae_df.drop("Total_Sales")

In [0]:
america_df.write.format("delta").mode("overwrite").saveAsTable("azure_wak_learn.silver.america_sales") 
india_df.write.format("delta").mode("overwrite").saveAsTable("azure_wak_learn.silver.india_sales") 
uae_df.write.format("delta").mode("overwrite").saveAsTable("azure_wak_learn.silver.uae_sale")